# ClusterBP v2: первые события

Основная модель в основном предсказывает повторение: в `additional/first_episodes.csv` её чувствительность
на первых эпизодах — 0.02 (A) и 0.04 (B), тогда как правило «максимальный САД» ловит 0.06 и 0.17.
История событий — самый сильный признак, и на окнах без событий в прошлом модели не на что опереться.

Здесь — отдельная постановка для пациентов **без событий этого типа до отсечки** («наивные» окна):

- **подмножество:** окна, у которых `ev_{t}_days_since` пусто, то есть до отсечки у пациента не было ни одного события типа t.
  Строже — окна без событий A, B и OLD вообще (`без тревог АД`);
- **модели:** правила по давлению; XGB_full, обученный на всех окнах; XGBoost и логистическая регрессия,
  обученные **только на наивных окнах**; то же с признаками длинной истории (30–90 дней, `clusterbp_extra.long_history_features`);
- **оценка:** только на наивных окнах тестовых пациентов, при их собственной доле событий; пороги — на наивных окнах
  внутренней валидации; сравнение с правилом «максимальный САД» — парный бутстреп по пациентам;
- **рабочие точки:** чувствительность при специфичности 90% и 95% (одинаковая нагрузка тревогами для всех моделей)
  и доля первых эпизодов пациента, перед которыми была хотя бы одна тревога.

Фолды и фенотипы — как в `clusterbp_v2.ipynb`.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from clusterbp_lib import RESULTS_DIR, DATA_DIR, MODELS_DIR, SEED, TARGETS, FEATURE_SETS, BP_COLS, STATIC_COLS, N_FOLDS, \
    fold_masks, best_threshold, load_xgb
from clusterbp_extra import (load_windows, phenotypes, fit_predict, compare_table,
                             long_history_features)

warnings.filterwarnings('ignore')
OUT = os.path.join(RESULTS_DIR, 'first_events')
os.makedirs(OUT, exist_ok=True)
FE_TARGETS = ['A', 'B']

win, pheno_win, pheno_first, pheno_first_pid = load_windows()
meas = pd.read_parquet(os.path.join(DATA_DIR, 'measurements.parquet'))
events = pd.read_parquet(os.path.join(DATA_DIR, 'events.parquet'))
LH = long_history_features(win, meas, events)
# у наивных окон счётчики событий своего типа нулевые; события других типов и «все КЗС» остаются признаками
LH_BP = [c for c in LH.columns if '_ev_' not in c]
for c in LH.columns:
    win[c] = LH[c].values
pid = win['pid'].values

NAIVE = {t: np.isnan(win[f'ev_{t}_days_since'].values) for t in FE_TARGETS}
NO_ALERTS = np.isnan(win['ev_A_days_since'].values) & np.isnan(win['ev_B_days_since'].values) & \
            np.isnan(win['ev_OLD_days_since'].values)
rows = []
for t in FE_TARGETS:
    y = win[f'y_{t}'].values
    for label, m in [('все окна', np.ones(len(win), bool)), (f'без событий {t} до отсечки', NAIVE[t]),
                     ('без тревог АД (A, B, OLD) до отсечки', NO_ALERTS)]:
        rows.append({'target': t, 'подмножество': label, 'окон': int(m.sum()), 'пациентов': len(np.unique(pid[m])),
                     'положительных окон': int(y[m].sum()), 'доля событий': y[m].mean()})
cohort = pd.DataFrame(rows)
cohort.to_csv(os.path.join(OUT, 'subsets.csv'), index=False)
cohort

In [ ]:
RULES = {'правило: макс. САД': 'sbp_max', 'правило: средний САД': 'sbp_mean', 'правило: доля САД ≥ 160': 'frac160',
         'правило: последний САД': 'sbp_last'}


def own_cols(t):
    """Признаки без счётчиков событий своего типа (на наивных окнах они постоянны)."""
    return [c for c in FEATURE_SETS['full'] if not c.startswith(f'ev_{t}_')]


MODELS = {
    'XGB_full (обучен на всех окнах)': None,
    'XGB, обучен на наивных окнах': lambda t: own_cols(t),
    'XGB на наивных + длинная история': lambda t: own_cols(t) + LH_BP + [c for c in LH.columns
                                                                     if '_ev_' in c and f'_ev_{t}' not in c],
    'LR на наивных (только АД и статика)': 'lr',
}
scores = {t: {m: (np.full(len(win), np.nan), np.zeros(len(win), np.int8)) for m in list(MODELS) + list(RULES)}
          for t in FE_TARGETS}
t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    phenotypes(win, pheno_win, pheno_first, pheno_first_pid, k)
    for t in FE_TARGETS:
        y = win[f'y_{t}'].values
        nv = NAIVE[t]
        res = {}
        mpath = os.path.join(MODELS_DIR, f'xgb_full_{t}_fold{k}.json')
        if os.path.exists(mpath):      # модель фолда из clusterbp_v2.ipynb
            s = load_xgb('full', t, k).predict_proba(win[FEATURE_SETS['full']].values)[:, 1]
        else:
            s, _, _ = fit_predict(win, FEATURE_SETS['full'], y, tr, va, SEED + k)
        res['XGB_full (обучен на всех окнах)'] = s
        for name, spec in MODELS.items():
            if spec is None:
                continue
            if spec == 'lr':
                cols = BP_COLS + STATIC_COLS
                lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=3000))
                lr.fit(win.loc[tr & nv, cols].values, y[tr & nv])
                res[name] = lr.predict_proba(win[cols].values)[:, 1]
            else:
                s, _, _ = fit_predict(win, spec(t), y, tr & nv, va & nv, SEED + k)
                res[name] = s
        for name, col in RULES.items():
            res[name] = win[col].values.astype(float)
        for name, s in res.items():
            thr, _ = best_threshold(y[va & nv], np.nan_to_num(s[va & nv], nan=-1e9))
            sc, pr = scores[t][name]
            sc[te] = s[te]
            pr[te] = s[te] > thr
    print(f'фолд {k}: {(time.time() - t_all) / 60:.1f} мин')

In [ ]:
def sens_at_spec(y, s, spec):
    s = np.nan_to_num(s, nan=-1e9)
    thr = np.quantile(s[y == 0], spec)
    return (s[y == 1] > thr).mean()


tables = []
for t in FE_TARGETS:
    y = win[f'y_{t}'].values
    for subset, m in [(f'без событий {t}', NAIVE[t]), ('без тревог АД', NO_ALERTS)]:
        ent = {name: (m, y, np.nan_to_num(sc, nan=-1e9), pr) for name, (sc, pr) in scores[t].items()}
        tab = compare_table(ent, pid, ref='правило: макс. САД')
        for spec in [0.90, 0.95]:
            tab[f'чувствительность при специфичности {int(spec * 100)}%'] = [
                sens_at_spec(y[m], scores[t][n][0][m], spec) for n in tab.index]
        tab.insert(0, 'подмножество', subset)
        tab.insert(0, 'target', t)
        tables.append(tab)
fe = pd.concat(tables)
fe.to_csv(os.path.join(OUT, 'first_events_metrics.csv'))
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
show = ['target', 'подмножество', 'доля событий', 'AUROC', 'AUPRC', 'AUPRC / доля', 'AUPRC 95% ДИ', 'ΔAUPRC',
        'ΔAUPRC 95% ДИ', 'p ΔAUPRC', 'sensitivity', 'чувствительность при специфичности 95%']
display(fe[show].round(3))

## Первые эпизоды пациента

Эпизод — день с событием типа t. Первый эпизод пациента замечен, если хотя бы в одном окне, горизонт которого его
покрывает (отсечка в `(t_эп − gap − 7; t_эп − gap]`), была тревога. Все такие окна — наивные по определению.
Нагрузка — тревог на пациента в неделю среди наивных окон (пересчёт раз в 3 дня).

In [ ]:
def first_episode_detection(t, pred):
    cfg = TARGETS[t]
    ev = events[events.code.isin(cfg['codes'])]
    first_ev = ev.groupby('pid')['td'].min()
    order = np.lexsort((win['cut'].values, pid))
    w_pid, w_cut, w_pred = pid[order], win['cut'].values[order], pred[order]
    u, a_idx = np.unique(w_pid, return_index=True)
    bounds = dict(zip(u, zip(a_idx, np.r_[a_idx[1:], len(w_pid)])))
    hits, leads = [], []
    for p_, te_ in first_ev.items():
        if p_ not in bounds:
            continue
        a, b = bounds[p_]
        cuts = w_cut[a:b]
        lo = np.searchsorted(cuts, te_ - cfg['gap'] - cfg['horizon'], side='right')
        hi = np.searchsorted(cuts, te_ - cfg['gap'], side='right')
        if hi <= lo:
            continue
        al = np.flatnonzero(w_pred[a:b][lo:hi] == 1)
        hits.append(len(al) > 0)
        leads.append(te_ - cuts[lo + al[0]] if len(al) else np.nan)
    return np.array(hits), np.array(leads)


ep_rows = []
for t in FE_TARGETS:
    nv = NAIVE[t]
    for name, (sc, pr) in scores[t].items():
        hits, leads = first_episode_detection(t, pr)
        ep_rows.append({'target': t, 'модель': name, 'первых эпизодов': len(hits), 'доля замеченных': hits.mean(),
                        'упреждение, дней (медиана)': np.nanmedian(leads) if hits.any() else np.nan,
                        'тревог на пациента в неделю (наивные окна)': pr[nv].mean() * 7 / 3})
ep = pd.DataFrame(ep_rows)
ep.to_csv(os.path.join(OUT, 'first_episode_detection.csv'), index=False)
display(ep.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, t in zip(axes, FE_TARGETS):
    sub = fe[(fe.target == t) & (fe['подмножество'] == f'без событий {t}')].sort_values('AUPRC')
    ax.barh(sub.index, sub['AUPRC'], color=['tab:gray' if n.startswith('правило') else 'tab:blue' for n in sub.index])
    ax.axvline(sub['доля событий'].iloc[0], color='k', ls=':', lw=1, label='доля событий')
    ax.set_title(f'{t}: окна без событий {t} до отсечки')
    ax.set_xlabel('AUPRC')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'first_events_auprc.png'), dpi=200, bbox_inches='tight')
plt.show()